In [1]:
import re
import torch
import joblib

from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

In [2]:
def normalize_text(text):

    text = text.lower()

    slang_dict = {
        "khong": "không",
        "ko": "không",
        "k": "không",
        "hok": "không",
        "giong": "giống",
        "sp": "sản phẩm",
        "mik": "mình",
        "dc": "được",
        "đc": "được"
    }

    for k, v in slang_dict.items():
        text = text.replace(k, v)

    text = re.sub(r'(.)\1+', r'\1', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [3]:
import urllib.request
import joblib

urllib.request.urlretrieve(
    "https://drive.google.com/uc?export=download&id=1JzLvv0eehCJ9m7XP2XUnaR4rkJqr5xav",
    "lr_model.pkl"
)

urllib.request.urlretrieve(
    "https://drive.google.com/uc?export=download&id=19QU-npprxOsaVSDKDmb7Pir5RU3uCUB6",
    "vectorizer.pkl"
)

lr_model = joblib.load("lr_model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

print("✅ Loaded Logistic Regression model!")


✅ Loaded Logistic Regression model!


In [4]:
def predict_lr(text):

    text = normalize_text(text)

    vec = vectorizer.transform([text])

    pred = lr_model.predict(vec)[0]

    reverse_label_map = {
        0: "NEG",
        1: "NEU",
        2: "POS"
    }

    return reverse_label_map[pred]


In [5]:
# 5️⃣ Load PhoBERT model
# =========================

!pip install -q gdown transformers

import gdown
from transformers import AutoTokenizer

folder_url = "https://drive.google.com/drive/folders/1GzNM3rhTIB1nfoPAxxv37XR2lkZgdCPj?usp=sharing"

gdown.download_folder(folder_url, output="my_tokenizer", quiet=False, use_cookies=False)

# load tokenizer từ folder local
tokenizer = AutoTokenizer.from_pretrained("my_tokenizer")

phobert_model = AutoModelForSequenceClassification.from_pretrained(
    model_path
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

phobert_model.to(device)

phobert_model.eval()

print("✅ Loaded PhoBERT model!")

Retrieving folder contents
Failed to retrieve folder contents
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


OSError: my_tokenizer is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
def predict_phobert(text):

    text = normalize_text(text)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = phobert_model(**inputs)

        logits = outputs.logits

        pred_class = torch.argmax(
            logits,
            dim=1
        ).item()

    reverse_label_map = {
        0: "NEG",
        1: "NEU",
        2: "POS"
    }

    return reverse_label_map[pred_class]


In [ ]:

examples = [

    "Sản phẩm này rất tốt",

    "không giống ảnh",

    "mắc rất khó chịu",

    "tạm chấp nhận đc nhưng k ấn tượng lắm",

    "vải đẹp, da mềm có hương thơm nhma khi ship đến thì mẫu k giống shop đã đưa lên",

    "sương sương",

    "quá tệ",
    "Sp này mik cảm thấy thật sự chưa tuyệt vời lắm",
    "Sp này mik cảm thấy k tệ"

]


print("\n=== Logistic Regression Predictions ===")

for text in examples:

    print(text, "->", predict_lr(text))


print("\n=== PhoBERT Predictions ===")

for text in examples:

    print(text, "->", predict_phobert(text))